## Libraries

In [ ]:
!pip uninstall yfinance
!pip uninstall  scikit-learn
!pip uninstall  tensorflow
!pip uninstall  vaderSentiment
!pip uninstall  matplotlib

In [3]:
!pip install yfinance==0.2.37
!pip install scikit-learn==1.4.2
!pip install tensorflow==2.16.1
!pip install vaderSentiment==3.3.2
!pip install matplotlib==3.8.4

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


ModuleNotFoundError: No module named 'tensorflow.keras'

## Load Data

In [2]:
# Step 1: Fetch historical stock price data
ticker = "MSFT"
data = yf.download(ticker, start="2024-04-01", end="2025-04-01")
closing_prices = data['Close'].values.reshape(-1, 1)

# Step 2: Fetch financial news + sentiment (Alpha Vantage)
api_key = "RYEJYRK0OTTEPD0V"

url = (
    f"https://www.alphavantage.co/query?"
    f"function=NEWS_SENTIMENT&tickers={ticker}"
    f"&time_from=20240401T0000&time_to=20250401T0000"
    f"&apikey={api_key}"
)

response = requests.get(url)
news_json = response.json()

print("Raw API response keys:", news_json.keys())

articles = news_json.get("feed", [])
df_news = pd.DataFrame(articles)

if not df_news.empty:
    df_news["date"] = pd.to_datetime(df_news["time_published"]).dt.date
    df_news["sentiment"] = pd.to_numeric(df_news["overall_sentiment_score"], errors="coerce")

    # Average sentiment per day
    daily_sentiment = df_news.groupby("date")["sentiment"].mean()

    print("\nSample news data with API sentiment:")
    print(df_news[["date", "title", "overall_sentiment_score", "sentiment"]].head(10))

    print("\nDaily sentiment sample:")
    print(daily_sentiment.head(10))

    print("\nNews coverage check:")
    print("Earliest news date:", df_news["date"].min())
    print("Latest news date:", df_news["date"].max())
    print("Number of unique news dates:", df_news["date"].nunique())
else:
    daily_sentiment = pd.Series(dtype=float)
    print("No news articles returned from Alpha Vantage.")

# Align sentiment with stock dates
stock_dates = data.index.date
sentiments = []

for d in stock_dates:
    sentiments.append(daily_sentiment.get(d, 0))  # 0 if no news for that date

sentiments = np.array(sentiments).reshape(-1, 1)

print("\nStock dates sample:", list(stock_dates[:5]))
print("News dates sample:", list(daily_sentiment.index[:5]))
print("Sentiment array shape:", sentiments.shape)
print("First 10 sentiment values:", sentiments[:10])

# Optional: view first few aligned stock/sentiment rows
debug_df = pd.DataFrame({
    "date": stock_dates,
    "close_price": closing_prices.flatten(),
    "sentiment": sentiments.flatten()
})

print("\nAligned stock + sentiment sample:")
print(debug_df.head(10))

# Step 3: Scale data
price_scaler = MinMaxScaler()
sentiment_scaler = MinMaxScaler()

scaled_prices = price_scaler.fit_transform(closing_prices)
scaled_sentiments = sentiment_scaler.fit_transform(sentiments)

# Step 4: Sliding window
window_size = 60
X, y = [], []

for i in range(window_size, len(scaled_prices)):
    X.append(np.column_stack((
        scaled_prices[i - window_size:i, 0],
        scaled_sentiments[i - window_size:i, 0]
    )))
    y.append(scaled_prices[i, 0])

X = np.array(X)
y = np.array(y)

print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)

# Step 5: Train/test split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# Step 6: Build LSTM model
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(window_size, 2)))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

# Step 7: Train model
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

# Step 8: Make predictions
predictions = model.predict(X_test)

# Step 9: Inverse transform
predictions = price_scaler.inverse_transform(predictions)
y_test_actual = price_scaler.inverse_transform(y_test.reshape(-1, 1))

# Step 10: Evaluate performance
mae = np.mean(np.abs(predictions - y_test_actual))
mse = mean_squared_error(y_test_actual, predictions)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test_actual - predictions) / y_test_actual)) * 100

print("\nModel Performance for MSFT:")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")


results = pd.DataFrame({
    "Stock": ["Microsoft"],
    "MAE": [round(mae, 2)],
    "MSE": [round(mse, 2)],
    "RMSE": [round(rmse, 2)],
    "MAPE (%)": [round(mape, 2)]
})

print("\nResults Table:")
print(results)

Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%%**********************]  1 of 1 completed

1 Failed download:
['MSFT']: Exception('%ticker%: No timezone found, symbol may be delisted')


NameError: name 'requests' is not defined

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("\nFULL API NEWS DATA:")
print(df_news[["time_published", "title", "overall_sentiment_score"]])